In [2]:
exact Siamese U-Net architecture
exact bias=False convolution setup
exact checkpoint format
uses your existing cached validation tiles
automatically finds the original LEVIR-CD test A/B/label folders
crops test images into 256×256 tiles
evaluates the best checkpoint
finds the best threshold on validation only
evaluates the test set using both 0.50 and the validation-selected threshold
reports IoU, Dice/F1, precision, recall, pixel accuracy
reports predicted vs ground-truth change %
creates a CSV with threshold sweep results
creates a confusion matrix
saves qualitative prediction examples
does not touch your training checkpoint

SyntaxError: invalid character '×' (U+00D7) (2694647222.py, line 6)

In [6]:
# ============================================================
# LEVIR-CD — FINAL TEST EVALUATION
# Siamese U-Net Change Detection
# ============================================================

import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ============================================================
# 1. PATHS
# ============================================================

BASE_DIR = Path(
    r"C:\College\Dissertation\Data\FINAL_LEVIR_DATASET"
)

# ------------------------------------------------------------
# ACTUAL LEVIR-CD TEST DATA
# 001 = train
# 002 = test
# ------------------------------------------------------------

LEVIR_CD_TEST_ROOT = Path(
    r"C:\College\Dissertation\Data\LEVIR-CD"
    r"\LEVIR-CD-20261002T144955Z-1-002"
    r"\LEVIR-CD"
)

TEST_A = LEVIR_CD_TEST_ROOT / "test" / "A"
TEST_B = LEVIR_CD_TEST_ROOT / "test" / "B"
TEST_MASK = LEVIR_CD_TEST_ROOT / "test" / "label"

# ------------------------------------------------------------
# TRAINED MODEL
# ------------------------------------------------------------

CHECKPOINT = (
    BASE_DIR
    / "CHANGE_DETECTION_MODEL"
    / "best_change_detection_model.pth"
)

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_DIR = (
    BASE_DIR
    / "CHANGE_DETECTION_MODEL"
    / "FINAL_TEST_EVALUATION"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. PRINT / VERIFY PATHS
# ============================================================

print("=" * 70)
print("LEVIR-CD — FINAL TEST EVALUATION")
print("=" * 70)

print("\nTest root:")
print(LEVIR_CD_TEST_ROOT)

print("\nA:")
print(TEST_A)

print("\nB:")
print(TEST_B)

print("\nLabel:")
print(TEST_MASK)

print("\nCheckpoint:")
print(CHECKPOINT)


# ============================================================
# 3. CHECK PATHS
# ============================================================

for path in [
    TEST_A,
    TEST_B,
    TEST_MASK,
    CHECKPOINT
]:

    if not path.exists():

        raise FileNotFoundError(
            f"\nRequired path does not exist:\n{path}"
        )

print("\n✓ All required paths found.")


# ============================================================
# 4. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:", device)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# 5. MODEL ARCHITECTURE
# ============================================================

class DoubleConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            )
        )

    def forward(self, x):

        return self.block(x)


class SiameseUNet(nn.Module):

    def __init__(self):

        super().__init__()

        # ----------------------------------------------------
        # Shared encoder
        # ----------------------------------------------------

        self.enc1 = DoubleConv(
            3,
            32
        )

        self.pool1 = nn.MaxPool2d(
            2
        )

        self.enc2 = DoubleConv(
            32,
            64
        )

        self.pool2 = nn.MaxPool2d(
            2
        )

        self.enc3 = DoubleConv(
            64,
            128
        )

        self.pool3 = nn.MaxPool2d(
            2
        )

        self.enc4 = DoubleConv(
            128,
            256
        )

        # ----------------------------------------------------
        # Decoder
        # ----------------------------------------------------

        self.up3 = nn.ConvTranspose2d(
            256,
            128,
            kernel_size=2,
            stride=2
        )

        self.dec3 = DoubleConv(
            256,
            128
        )

        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=2,
            stride=2
        )

        self.dec2 = DoubleConv(
            128,
            64
        )

        self.up1 = nn.ConvTranspose2d(
            64,
            32,
            kernel_size=2,
            stride=2
        )

        self.dec1 = DoubleConv(
            64,
            32
        )

        # ----------------------------------------------------
        # Segmentation head
        # ----------------------------------------------------

        self.mask_head = nn.Conv2d(
            32,
            1,
            kernel_size=1
        )


    def encode(self, x):

        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool1(e1)
        )

        e3 = self.enc3(
            self.pool2(e2)
        )

        e4 = self.enc4(
            self.pool3(e3)
        )

        return e1, e2, e3, e4


    def forward(
        self,
        img_a,
        img_b
    ):

        a1, a2, a3, a4 = self.encode(
            img_a
        )

        b1, b2, b3, b4 = self.encode(
            img_b
        )

        # ----------------------------------------------------
        # Absolute feature differences
        # ----------------------------------------------------

        d1 = torch.abs(
            a1 - b1
        )

        d2 = torch.abs(
            a2 - b2
        )

        d3 = torch.abs(
            a3 - b3
        )

        d4 = torch.abs(
            a4 - b4
        )

        # ----------------------------------------------------
        # Decoder
        # ----------------------------------------------------

        x = self.up3(d4)

        x = torch.cat(
            [x, d3],
            dim=1
        )

        x = self.dec3(x)

        x = self.up2(x)

        x = torch.cat(
            [x, d2],
            dim=1
        )

        x = self.dec2(x)

        x = self.up1(x)

        x = torch.cat(
            [x, d1],
            dim=1
        )

        x = self.dec1(x)

        return self.mask_head(x)


# ============================================================
# 6. LOAD TRAINED MODEL
# ============================================================

print("\nLoading trained model...")

checkpoint = torch.load(
    CHECKPOINT,
    map_location=device
)

model = SiameseUNet().to(
    device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("✓ Model loaded successfully.")

print(
    "Checkpoint epoch:",
    checkpoint.get("epoch")
)

print(
    "Best validation IoU:",
    checkpoint.get("best_iou")
)


# ============================================================
# 7. FIND TEST IMAGES
# ============================================================

def get_image_files(folder):

    extensions = [
        "*.png",
        "*.jpg",
        "*.jpeg",
        "*.tif",
        "*.tiff"
    ]

    files = []

    for ext in extensions:

        files.extend(
            folder.glob(ext)
        )

    return sorted(files)


A_files = get_image_files(
    TEST_A
)

B_files = get_image_files(
    TEST_B
)

MASK_files = get_image_files(
    TEST_MASK
)

print("\nOriginal LEVIR-CD test images:")

print(
    "A     :",
    len(A_files)
)

print(
    "B     :",
    len(B_files)
)

print(
    "Label :",
    len(MASK_files)
)


# ============================================================
# 8. MATCH A / B / LABEL FILES
# ============================================================

B_dict = {
    p.stem: p
    for p in B_files
}

MASK_dict = {
    p.stem: p
    for p in MASK_files
}


matched = []

for a_path in A_files:

    stem = a_path.stem

    if stem not in B_dict:
        continue

    if stem not in MASK_dict:
        continue

    matched.append(
        (
            a_path,
            B_dict[stem],
            MASK_dict[stem]
        )
    )


print(
    "\nMatched image pairs:",
    len(matched)
)


if len(matched) == 0:

    raise RuntimeError(
        "No matching A/B/label test pairs found."
    )


# ============================================================
# 9. CREATE TEST TILE CACHE
# ============================================================

CACHE_ROOT = (
    BASE_DIR
    / "CHANGE_DETECTION_MODEL"
    / "TEST_256_CACHE"
)

CACHE_A = CACHE_ROOT / "A"
CACHE_B = CACHE_ROOT / "B"
CACHE_MASK = CACHE_ROOT / "mask"

CACHE_A.mkdir(
    parents=True,
    exist_ok=True
)

CACHE_B.mkdir(
    parents=True,
    exist_ok=True
)

CACHE_MASK.mkdir(
    parents=True,
    exist_ok=True
)


TILE_SIZE = 256

records = []

print(
    "\nCreating 256×256 test tiles..."
)


for pair_idx, (
    a_path,
    b_path,
    mask_path
) in enumerate(matched):

    stem = a_path.stem

    img_a = Image.open(
        a_path
    ).convert("RGB")

    img_b = Image.open(
        b_path
    ).convert("RGB")

    mask = Image.open(
        mask_path
    ).convert("L")


    # --------------------------------------------------------
    # Verify original dimensions
    # --------------------------------------------------------

    if img_a.size != (1024, 1024):

        raise ValueError(
            f"Unexpected A size: "
            f"{a_path} -> {img_a.size}"
        )

    if img_b.size != (1024, 1024):

        raise ValueError(
            f"Unexpected B size: "
            f"{b_path} -> {img_b.size}"
        )

    if mask.size != (1024, 1024):

        raise ValueError(
            f"Unexpected mask size: "
            f"{mask_path} -> {mask.size}"
        )


    # --------------------------------------------------------
    # 4 × 4 = 16 exact tiles
    # --------------------------------------------------------

    for row in range(4):

        for col in range(4):

            tile_id = (
                row * 4 + col
            )

            left = (
                col * TILE_SIZE
            )

            top = (
                row * TILE_SIZE
            )

            right = (
                left + TILE_SIZE
            )

            bottom = (
                top + TILE_SIZE
            )


            tile_a = img_a.crop(
                (
                    left,
                    top,
                    right,
                    bottom
                )
            )

            tile_b = img_b.crop(
                (
                    left,
                    top,
                    right,
                    bottom
                )
            )

            tile_mask = mask.crop(
                (
                    left,
                    top,
                    right,
                    bottom
                )
            )


            filename = (
                f"{stem}_tile_{tile_id:02d}.png"
            )


            tile_a.save(
                CACHE_A / filename
            )

            tile_b.save(
                CACHE_B / filename
            )

            tile_mask.save(
                CACHE_MASK / filename
            )


            records.append({

                "filename":
                    filename,

                "parent":
                    stem,

                "row":
                    row,

                "col":
                    col,

                "tile":
                    tile_id
            })


test_df = pd.DataFrame(
    records
)


test_df.to_csv(
    CACHE_ROOT /
    "test_tiles.csv",
    index=False
)


print(
    "\n✓ Test tiles created:",
    len(test_df)
)

print(
    "Expected:",
    len(matched) * 16
)


# ============================================================
# 10. TEST DATASET
# ============================================================

class TestDataset(Dataset):

    def __init__(
        self,
        dataframe,
        cache_root
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
        )

        self.cache_root = Path(
            cache_root
        )


    def __len__(self):

        return len(self.df)


    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        filename = row["filename"]


        img_a = Image.open(
            self.cache_root /
            "A" /
            filename
        ).convert("RGB")


        img_b = Image.open(
            self.cache_root /
            "B" /
            filename
        ).convert("RGB")


        mask = Image.open(
            self.cache_root /
            "mask" /
            filename
        ).convert("L")


        # ----------------------------------------------------
        # Convert images to [0,1]
        # ----------------------------------------------------

        img_a = np.asarray(
            img_a,
            dtype=np.float32
        ) / 255.0

        img_b = np.asarray(
            img_b,
            dtype=np.float32
        ) / 255.0


        # ----------------------------------------------------
        # Binary mask
        # ----------------------------------------------------

        mask = np.asarray(
            mask,
            dtype=np.float32
        )

        mask = (
            mask > 127
        ).astype(
            np.float32
        )


        # HWC → CHW

        img_a = torch.from_numpy(
            img_a.transpose(2, 0, 1)
        )

        img_b = torch.from_numpy(
            img_b.transpose(2, 0, 1)
        )

        mask = torch.from_numpy(
            mask[None, :, :]
        )


        return (
            img_a,
            img_b,
            mask,
            filename
        )


# ============================================================
# 11. DATALOADER
# ============================================================

dataset = TestDataset(
    test_df,
    CACHE_ROOT
)

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(
    "\nTest tiles:",
    len(dataset)
)

print(
    "Test batches:",
    len(loader)
)


# ============================================================
# 12. RUN TEST INFERENCE
# ============================================================

all_probs = []
all_targets = []
all_names = []

start_time = time.time()


with torch.no_grad():

    for batch_idx, (
        img_a,
        img_b,
        masks,
        filenames
    ) in enumerate(loader):

        img_a = img_a.to(
            device,
            non_blocking=True
        )

        img_b = img_b.to(
            device,
            non_blocking=True
        )


        logits = model(
            img_a,
            img_b
        )


        probs = torch.sigmoid(
            logits
        )


        all_probs.append(
            probs.cpu().numpy()
        )

        all_targets.append(
            masks.numpy()
        )

        all_names.extend(
            filenames
        )


        if (
            batch_idx + 1
        ) % 20 == 0:

            print(
                f"Processed "
                f"{batch_idx + 1}/"
                f"{len(loader)} batches"
            )


all_probs = np.concatenate(
    all_probs,
    axis=0
)

all_targets = np.concatenate(
    all_targets,
    axis=0
)


elapsed = (
    time.time() -
    start_time
)


print(
    "\n✓ Test inference complete."
)

print(
    f"Time: {elapsed / 60:.2f} minutes"
)

print(
    "Probability shape:",
    all_probs.shape
)

print(
    "Target shape:",
    all_targets.shape
)


# ============================================================
# 13. METRIC FUNCTION
# ============================================================

def calculate_metrics(
    probabilities,
    targets,
    threshold
):

    predictions = (
        probabilities >= threshold
    ).astype(np.uint8)

    targets = (
        targets > 0.5
    ).astype(np.uint8)


    tp = np.logical_and(
        predictions == 1,
        targets == 1
    ).sum()

    tn = np.logical_and(
        predictions == 0,
        targets == 0
    ).sum()

    fp = np.logical_and(
        predictions == 1,
        targets == 0
    ).sum()

    fn = np.logical_and(
        predictions == 0,
        targets == 1
    ).sum()


    iou = (
        tp /
        (tp + fp + fn)
        if (tp + fp + fn) > 0
        else 0
    )


    dice = (
        2 * tp /
        (2 * tp + fp + fn)
        if (2 * tp + fp + fn) > 0
        else 0
    )


    precision = (
        tp /
        (tp + fp)
        if (tp + fp) > 0
        else 0
    )


    recall = (
        tp /
        (tp + fn)
        if (tp + fn) > 0
        else 0
    )


    pixel_accuracy = (
        (tp + tn) /
        (tp + tn + fp + fn)
    )


    predicted_change_percent = (
        predictions.mean() * 100
    )


    gt_change_percent = (
        targets.mean() * 100
    )


    return {

        "threshold":
            threshold,

        "IoU":
            iou,

        "Dice":
            dice,

        "Precision":
            precision,

        "Recall":
            recall,

        "Pixel_Accuracy":
            pixel_accuracy,

        "Predicted_Change_Percent":
            predicted_change_percent,

        "GT_Change_Percent":
            gt_change_percent,

        "TP":
            int(tp),

        "TN":
            int(tn),

        "FP":
            int(fp),

        "FN":
            int(fn)
    }


# ============================================================
# 14. FINAL TEST EVALUATION
# ============================================================

# 0.50 = standard threshold
# 0.85 = selected from VALIDATION SET

TEST_THRESHOLDS = [
    0.50,
    0.85
]


results = []


for threshold in TEST_THRESHOLDS:

    metrics = calculate_metrics(
        all_probs,
        all_targets,
        threshold
    )

    results.append(
        metrics
    )


results_df = pd.DataFrame(
    results
)


# ============================================================
# 15. PRINT FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL LEVIR-CD TEST RESULTS")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


# ============================================================
# 16. SAVE RESULTS
# ============================================================

results_df.to_csv(
    OUTPUT_DIR /
    "test_metrics.csv",
    index=False
)


np.save(
    OUTPUT_DIR /
    "test_probabilities.npy",
    all_probs
)


np.save(
    OUTPUT_DIR /
    "test_targets.npy",
    all_targets
)


test_df.to_csv(
    OUTPUT_DIR /
    "test_tile_metadata.csv",
    index=False
)


print(
    "\n✓ Saved:"
)

print(
    OUTPUT_DIR /
    "test_metrics.csv"
)


# ============================================================
# 17. SAVE CONFUSION MATRICES
# ============================================================

for threshold in TEST_THRESHOLDS:

    metrics = calculate_metrics(
        all_probs,
        all_targets,
        threshold
    )


    confusion_matrix = np.array([

        [
            metrics["TN"],
            metrics["FP"]
        ],

        [
            metrics["FN"],
            metrics["TP"]
        ]

    ])


    np.savetxt(

        OUTPUT_DIR /
        f"confusion_matrix_{threshold:.2f}.csv",

        confusion_matrix,

        delimiter=",",

        fmt="%d"
    )


# ============================================================
# 18. QUALITATIVE RESULTS
# ============================================================

VIS_DIR = (
    OUTPUT_DIR /
    "qualitative"
)

VIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Find changed and unchanged samples
# ------------------------------------------------------------

gt_change_ratio = (
    all_targets
    .reshape(len(all_targets), -1)
    .mean(axis=1)
)


changed_indices = np.where(
    gt_change_ratio > 0
)[0]


unchanged_indices = np.where(
    gt_change_ratio == 0
)[0]


selected_indices = []


selected_indices.extend(
    changed_indices[:10].tolist()
)

selected_indices.extend(
    unchanged_indices[:5].tolist()
)


selected_indices = (
    selected_indices[:15]
)


# ------------------------------------------------------------
# Generate visualizations
# ------------------------------------------------------------

for idx in selected_indices:

    filename = all_names[idx]


    img_a = np.asarray(
        Image.open(
            CACHE_A / filename
        ).convert("RGB")
    )


    img_b = np.asarray(
        Image.open(
            CACHE_B / filename
        ).convert("RGB")
    )


    gt = all_targets[
        idx,
        0
    ]


    prediction = (
        all_probs[
            idx,
            0
        ] >= 0.85
    ).astype(np.uint8)


    fig, axes = plt.subplots(
        1,
        4,
        figsize=(16, 4)
    )


    axes[0].imshow(
        img_a
    )

    axes[0].set_title(
        "Image A"
    )

    axes[0].axis(
        "off"
    )


    axes[1].imshow(
        img_b
    )

    axes[1].set_title(
        "Image B"
    )

    axes[1].axis(
        "off"
    )


    axes[2].imshow(
        gt,
        cmap="gray"
    )

    axes[2].set_title(
        "Ground Truth"
    )

    axes[2].axis(
        "off"
    )


    axes[3].imshow(
        prediction,
        cmap="gray"
    )

    axes[3].set_title(
        "Prediction (τ=0.85)"
    )

    axes[3].axis(
        "off"
    )


    plt.tight_layout()


    output_file = (
        VIS_DIR /
        filename.replace(
            ".png",
            "_evaluation.png"
        )
    )


    plt.savefig(
        output_file,
        dpi=150,
        bbox_inches="tight"
    )


    plt.close()


print(
    "\n✓ Qualitative visualizations saved:"
)

print(
    VIS_DIR
)


# ============================================================
# 19. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)


for _, row in results_df.iterrows():

    print(
        f"\nThreshold: "
        f"{row['threshold']:.2f}"
    )

    print(
        f"  IoU:        "
        f"{row['IoU']:.4f}"
    )

    print(
        f"  Dice:       "
        f"{row['Dice']:.4f}"
    )

    print(
        f"  Precision:  "
        f"{row['Precision']:.4f}"
    )

    print(
        f"  Recall:     "
        f"{row['Recall']:.4f}"
    )

    print(
        f"  Pixel Acc:  "
        f"{row['Pixel_Accuracy']:.4f}"
    )

    print(
        f"  Pred Change:"
        f" {row['Predicted_Change_Percent']:.4f}%"
    )

    print(
        f"  GT Change:  "
        f"{row['GT_Change_Percent']:.4f}%"
    )


print("\n")
print("=" * 70)
print("TEST EVALUATION COMPLETE")
print("=" * 70)

LEVIR-CD — FINAL TEST EVALUATION

Test root:
C:\College\Dissertation\Data\LEVIR-CD\LEVIR-CD-20261002T144955Z-1-002\LEVIR-CD

A:
C:\College\Dissertation\Data\LEVIR-CD\LEVIR-CD-20261002T144955Z-1-002\LEVIR-CD\test\A

B:
C:\College\Dissertation\Data\LEVIR-CD\LEVIR-CD-20261002T144955Z-1-002\LEVIR-CD\test\B

Label:
C:\College\Dissertation\Data\LEVIR-CD\LEVIR-CD-20261002T144955Z-1-002\LEVIR-CD\test\label

Checkpoint:
C:\College\Dissertation\Data\FINAL_LEVIR_DATASET\CHANGE_DETECTION_MODEL\best_change_detection_model.pth

✓ All required paths found.

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU

Loading trained model...
✓ Model loaded successfully.
Checkpoint epoch: 66
Best validation IoU: 0.769584989985579

Original LEVIR-CD test images:
A     : 128
B     : 128
Label : 128

Matched image pairs: 128

Creating 256×256 test tiles...

✓ Test tiles created: 2048
Expected: 2048

Test tiles: 2048
Test batches: 128
Processed 20/128 batches
Processed 40/128 batches
Processed 60/128 batches
Pro

In [1]:
# ============================================================
# LEVIR-CD — DETAILED ERROR ANALYSIS
# Uses already-generated TEST predictions
# NO RETRAINING
# ============================================================

import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt


# ============================================================
# 1. PATHS
# ============================================================

BASE_DIR = Path(
    r"C:\College\Dissertation\Data\FINAL_LEVIR_DATASET"
)

EVAL_DIR = (
    BASE_DIR
    / "CHANGE_DETECTION_MODEL"
    / "FINAL_TEST_EVALUATION"
)

CACHE_ROOT = (
    BASE_DIR
    / "CHANGE_DETECTION_MODEL"
    / "TEST_256_CACHE"
)

CACHE_A = CACHE_ROOT / "A"
CACHE_B = CACHE_ROOT / "B"
CACHE_MASK = CACHE_ROOT / "mask"

OUTPUT_DIR = (
    EVAL_DIR
    / "ERROR_ANALYSIS"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

VIS_DIR = OUTPUT_DIR / "VISUALIZATIONS"

VIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. FILES
# ============================================================

PROB_FILE = (
    EVAL_DIR
    / "test_probabilities.npy"
)

TARGET_FILE = (
    EVAL_DIR
    / "test_targets.npy"
)

METADATA_FILE = (
    EVAL_DIR
    / "test_tile_metadata.csv"
)


print("=" * 70)
print("LEVIR-CD — DETAILED ERROR ANALYSIS")
print("=" * 70)

for path in [
    PROB_FILE,
    TARGET_FILE,
    METADATA_FILE,
    CACHE_A,
    CACHE_B,
    CACHE_MASK
]:

    if not path.exists():

        raise FileNotFoundError(
            f"\nRequired file/folder not found:\n{path}"
        )

print("\n✓ All required files found.")


# ============================================================
# 3. LOAD DATA
# ============================================================

print("\nLoading saved predictions...")

probabilities = np.load(
    PROB_FILE
)

targets = np.load(
    TARGET_FILE
)

metadata = pd.read_csv(
    METADATA_FILE
)

print(
    "Probabilities:",
    probabilities.shape
)

print(
    "Targets:",
    targets.shape
)

print(
    "Metadata:",
    metadata.shape
)


# ============================================================
# 4. BASIC CONSISTENCY CHECK
# ============================================================

if len(probabilities) != len(targets):

    raise ValueError(
        "Probability and target sample counts differ."
    )

if len(probabilities) != len(metadata):

    raise ValueError(
        "Prediction count and metadata count differ."
    )

print(
    "\n✓ Prediction / target / metadata counts match."
)


# ============================================================
# 5. FLATTEN
# ============================================================

probs = probabilities[:, 0]

gt = (
    targets[:, 0] > 0.5
).astype(np.uint8)


# ============================================================
# 6. METRIC FUNCTION
# ============================================================

def calculate_metrics(
    prediction,
    target
):

    prediction = prediction.astype(
        np.uint8
    )

    target = target.astype(
        np.uint8
    )

    tp = np.logical_and(
        prediction == 1,
        target == 1
    ).sum()

    tn = np.logical_and(
        prediction == 0,
        target == 0
    ).sum()

    fp = np.logical_and(
        prediction == 1,
        target == 0
    ).sum()

    fn = np.logical_and(
        prediction == 0,
        target == 1
    ).sum()


    iou = (
        tp / (tp + fp + fn)
        if (tp + fp + fn) > 0
        else 0.0
    )

    dice = (
        2 * tp /
        (2 * tp + fp + fn)
        if (2 * tp + fp + fn) > 0
        else 0.0
    )

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    pixel_accuracy = (
        (tp + tn) /
        (tp + tn + fp + fn)
    )

    return {
        "IoU": iou,
        "Dice": dice,
        "Precision": precision,
        "Recall": recall,
        "Pixel_Accuracy": pixel_accuracy,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn)
    }


# ============================================================
# 7. OVERALL COMPARISON
# ============================================================

print("\n")
print("=" * 70)
print("OVERALL TEST COMPARISON")
print("=" * 70)

overall_results = []

for threshold in [0.50, 0.85]:

    prediction = (
        probs >= threshold
    ).astype(np.uint8)

    metrics = calculate_metrics(
        prediction,
        gt
    )

    metrics["Threshold"] = threshold

    metrics["Predicted_Change_Percent"] = (
        prediction.mean() * 100
    )

    metrics["GT_Change_Percent"] = (
        gt.mean() * 100
    )

    overall_results.append(
        metrics
    )


overall_df = pd.DataFrame(
    overall_results
)

print(
    overall_df[
        [
            "Threshold",
            "IoU",
            "Dice",
            "Precision",
            "Recall",
            "Pixel_Accuracy",
            "Predicted_Change_Percent",
            "GT_Change_Percent"
        ]
    ].to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


overall_df.to_csv(
    OUTPUT_DIR /
    "overall_comparison.csv",
    index=False
)


# ============================================================
# 8. PRIMARY ANALYSIS THRESHOLD
# ============================================================

THRESHOLD = 0.85

pred = (
    probs >= THRESHOLD
).astype(np.uint8)


# ============================================================
# 9. PER-TILE METRICS
# ============================================================

print("\n")
print("=" * 70)
print("CALCULATING PER-TILE METRICS")
print("=" * 70)


tile_rows = []


for i in range(len(probs)):

    tile_prob = probs[i]

    tile_gt = gt[i]

    tile_pred = (
        tile_prob >= THRESHOLD
    ).astype(np.uint8)


    metrics = calculate_metrics(
        tile_pred,
        tile_gt
    )


    gt_pixels = int(
        tile_gt.sum()
    )

    pred_pixels = int(
        tile_pred.sum()
    )


    total_pixels = tile_gt.size


    gt_change_percent = (
        gt_pixels /
        total_pixels *
        100
    )

    pred_change_percent = (
        pred_pixels /
        total_pixels *
        100
    )


    # --------------------------------------------------------
    # Error pixels
    # --------------------------------------------------------

    fp_pixels = (
        np.logical_and(
            tile_pred == 1,
            tile_gt == 0
        ).sum()
    )

    fn_pixels = (
        np.logical_and(
            tile_pred == 0,
            tile_gt == 1
        ).sum()
    )


    # --------------------------------------------------------
    # Completely missed change
    # --------------------------------------------------------

    completely_missed = (
        gt_pixels > 0
        and pred_pixels == 0
    )


    # --------------------------------------------------------
    # False-change tile
    # --------------------------------------------------------

    false_change = (
        gt_pixels == 0
        and pred_pixels > 0
    )


    # --------------------------------------------------------
    # No-change tile
    # --------------------------------------------------------

    no_change = (
        gt_pixels == 0
    )


    # --------------------------------------------------------
    # Change-size category
    # --------------------------------------------------------

    if gt_change_percent == 0:

        change_category = "No Change"

    elif gt_change_percent < 1:

        change_category = "Tiny (<1%)"

    elif gt_change_percent < 5:

        change_category = "Small (1-5%)"

    elif gt_change_percent < 15:

        change_category = "Medium (5-15%)"

    else:

        change_category = "Large (>15%)"


    row = metadata.iloc[i].to_dict()

    row.update({

        "GT_Change_Percent":
            gt_change_percent,

        "Predicted_Change_Percent":
            pred_change_percent,

        "GT_Changed_Pixels":
            gt_pixels,

        "Predicted_Changed_Pixels":
            pred_pixels,

        "FP_Pixels":
            int(fp_pixels),

        "FN_Pixels":
            int(fn_pixels),

        "Completely_Missed":
            completely_missed,

        "False_Change":
            false_change,

        "No_Change_GT":
            no_change,

        "Change_Category":
            change_category,

        "IoU":
            metrics["IoU"],

        "Dice":
            metrics["Dice"],

        "Precision":
            metrics["Precision"],

        "Recall":
            metrics["Recall"],

        "Pixel_Accuracy":
            metrics["Pixel_Accuracy"],

        "TP":
            metrics["TP"],

        "TN":
            metrics["TN"],

        "FP":
            metrics["FP"],

        "FN":
            metrics["FN"]
    })


    tile_rows.append(
        row
    )


tile_df = pd.DataFrame(
    tile_rows
)


tile_df.to_csv(
    OUTPUT_DIR /
    "per_tile_analysis.csv",
    index=False
)


print(
    "\n✓ Per-tile analysis saved."
)


# ============================================================
# 10. CHANGE-SIZE ANALYSIS
# ============================================================

print("\n")
print("=" * 70)
print("PERFORMANCE BY CHANGE SIZE")
print("=" * 70)


categories = [
    "No Change",
    "Tiny (<1%)",
    "Small (1-5%)",
    "Medium (5-15%)",
    "Large (>15%)"
]


category_rows = []


for category in categories:

    subset = tile_df[
        tile_df["Change_Category"]
        == category
    ]


    if len(subset) == 0:
        continue


    # For no-change tiles, IoU is tricky:
    # if both GT and prediction are empty,
    # the tile is considered correctly detected.
    valid_iou = subset["IoU"]


    category_rows.append({

        "Category":
            category,

        "Tiles":
            len(subset),

        "Mean_IoU":
            valid_iou.mean(),

        "Median_IoU":
            valid_iou.median(),

        "Mean_Dice":
            subset["Dice"].mean(),

        "Mean_Precision":
            subset["Precision"].mean(),

        "Mean_Recall":
            subset["Recall"].mean(),

        "Mean_GT_Change_Percent":
            subset["GT_Change_Percent"].mean(),

        "Mean_Pred_Change_Percent":
            subset[
                "Predicted_Change_Percent"
            ].mean(),

        "Completely_Missed":
            subset["Completely_Missed"].sum(),

        "False_Change":
            subset["False_Change"].sum()
    })


category_df = pd.DataFrame(
    category_rows
)


print(
    category_df.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.4f}"
    )
)


category_df.to_csv(
    OUTPUT_DIR /
    "performance_by_change_size.csv",
    index=False
)


# ============================================================
# 11. NO-CHANGE ANALYSIS
# ============================================================

print("\n")
print("=" * 70)
print("NO-CHANGE ANALYSIS")
print("=" * 70)


no_change_df = tile_df[
    tile_df["No_Change_GT"] == True
]


correct_no_change = (
    no_change_df[
        no_change_df[
            "Predicted_Changed_Pixels"
        ] == 0
    ]
)


false_change_df = (
    no_change_df[
        no_change_df[
            "Predicted_Changed_Pixels"
        ] > 0
    ]
)


print(
    "GT no-change tiles:",
    len(no_change_df)
)

print(
    "Correctly predicted no-change:",
    len(correct_no_change)
)

print(
    "False-change tiles:",
    len(false_change_df)
)

if len(no_change_df) > 0:

    print(
        "False-change rate:",
        f"{len(false_change_df) / len(no_change_df) * 100:.2f}%"
    )


false_change_df.to_csv(
    OUTPUT_DIR /
    "false_change_tiles.csv",
    index=False
)


# ============================================================
# 12. MISSED CHANGE ANALYSIS
# ============================================================

print("\n")
print("=" * 70)
print("MISSED-CHANGE ANALYSIS")
print("=" * 70)


change_tiles = tile_df[
    tile_df["GT_Changed_Pixels"] > 0
]


missed_tiles = tile_df[
    tile_df["Completely_Missed"] == True
]


print(
    "GT change tiles:",
    len(change_tiles)
)

print(
    "Completely missed:",
    len(missed_tiles)
)


if len(change_tiles) > 0:

    print(
        "Complete-miss rate:",
        f"{len(missed_tiles) / len(change_tiles) * 100:.2f}%"
    )


missed_tiles.to_csv(
    OUTPUT_DIR /
    "completely_missed_tiles.csv",
    index=False
)


# ============================================================
# 13. ERROR PIXEL ANALYSIS
# ============================================================

total_fp = int(
    tile_df["FP_Pixels"].sum()
)

total_fn = int(
    tile_df["FN_Pixels"].sum()
)

total_tp = int(
    tile_df["TP"].sum()
)

total_tn = int(
    tile_df["TN"].sum()
)


print("\n")
print("=" * 70)
print("PIXEL-LEVEL ERROR ANALYSIS")
print("=" * 70)

print(
    f"True Positives : {total_tp:,}"
)

print(
    f"True Negatives : {total_tn:,}"
)

print(
    f"False Positives: {total_fp:,}"
)

print(
    f"False Negatives: {total_fn:,}"
)


print(
    "\nFP / FN ratio:",
    f"{total_fp / max(total_fn, 1):.3f}"
)


# ============================================================
# 14. BEST / WORST TILES
# ============================================================

print("\n")
print("=" * 70)
print("BEST / WORST TILES")
print("=" * 70)


# Only meaningful change tiles
change_tile_df = tile_df[
    tile_df["GT_Changed_Pixels"] > 0
].copy()


best_tiles = (
    change_tile_df
    .sort_values(
        "IoU",
        ascending=False
    )
    .head(20)
)


worst_tiles = (
    change_tile_df
    .sort_values(
        "IoU",
        ascending=True
    )
    .head(20)
)


best_tiles.to_csv(
    OUTPUT_DIR /
    "best_20_change_tiles.csv",
    index=False
)


worst_tiles.to_csv(
    OUTPUT_DIR /
    "worst_20_change_tiles.csv",
    index=False
)


print("\nBest 10:")

print(
    best_tiles[
        [
            "filename",
            "GT_Change_Percent",
            "IoU",
            "Dice",
            "Precision",
            "Recall"
        ]
    ].head(10).to_string(
        index=False
    )
)


print("\nWorst 10:")

print(
    worst_tiles[
        [
            "filename",
            "GT_Change_Percent",
            "IoU",
            "Dice",
            "Precision",
            "Recall"
        ]
    ].head(10).to_string(
        index=False
    )
)


# ============================================================
# 15. VISUALIZATION FUNCTION
# ============================================================

def create_visualization(
    row,
    output_path,
    title_prefix
):

    filename = row["filename"]


    img_a = np.asarray(
        Image.open(
            CACHE_A / filename
        ).convert("RGB")
    )


    img_b = np.asarray(
        Image.open(
            CACHE_B / filename
        ).convert("RGB")
    )


    gt_mask = np.asarray(
        Image.open(
            CACHE_MASK / filename
        ).convert("L")
    )


    gt_mask = (
        gt_mask > 127
    ).astype(np.uint8)


    idx = metadata.index[
        metadata["filename"]
        == filename
    ]


    if len(idx) == 0:
        return


    idx = idx[0]


    probability = probs[idx]


    prediction = (
        probability >= THRESHOLD
    ).astype(np.uint8)


    # --------------------------------------------------------
    # Error map
    #
    # TP = white
    # FP = red
    # FN = blue
    # --------------------------------------------------------

    error_map = np.zeros(
        (
            256,
            256,
            3
        ),
        dtype=np.float32
    )


    tp = np.logical_and(
        prediction == 1,
        gt_mask == 1
    )

    fp = np.logical_and(
        prediction == 1,
        gt_mask == 0
    )

    fn = np.logical_and(
        prediction == 0,
        gt_mask == 1
    )


    error_map[tp] = [
        1.0,
        1.0,
        1.0
    ]

    error_map[fp] = [
        1.0,
        0.0,
        0.0
    ]

    error_map[fn] = [
        0.0,
        0.0,
        1.0
    ]


    metrics = calculate_metrics(
        prediction,
        gt_mask
    )


    fig, axes = plt.subplots(
        1,
        5,
        figsize=(20, 4)
    )


    axes[0].imshow(
        img_a
    )

    axes[0].set_title(
        "Image A"
    )

    axes[0].axis(
        "off"
    )


    axes[1].imshow(
        img_b
    )

    axes[1].set_title(
        "Image B"
    )

    axes[1].axis(
        "off"
    )


    axes[2].imshow(
        gt_mask,
        cmap="gray"
    )

    axes[2].set_title(
        "Ground Truth"
    )

    axes[2].axis(
        "off"
    )


    axes[3].imshow(
        prediction,
        cmap="gray"
    )

    axes[3].set_title(
        f"Prediction\nτ={THRESHOLD}"
    )

    axes[3].axis(
        "off"
    )


    axes[4].imshow(
        error_map
    )

    axes[4].set_title(
        "Error Map\n"
        "White=TP Red=FP Blue=FN"
    )

    axes[4].axis(
        "off"
    )


    fig.suptitle(
        f"{title_prefix}: {filename}\n"
        f"IoU={metrics['IoU']:.3f} | "
        f"Dice={metrics['Dice']:.3f} | "
        f"GT Change={gt_mask.mean()*100:.2f}%",
        fontsize=12
    )


    plt.tight_layout()


    plt.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight"
    )


    plt.close()


# ============================================================
# 16. VISUALIZE WORST TILES
# ============================================================

print("\n")
print("=" * 70)
print("GENERATING WORST-TILE VISUALIZATIONS")
print("=" * 70)


for rank, (_, row) in enumerate(
    worst_tiles.head(20).iterrows(),
    start=1
):

    output_file = (
        VIS_DIR /
        f"worst_{rank:02d}_{row['filename']}"
    )

    create_visualization(
        row,
        output_file,
        f"WORST #{rank}"
    )


print(
    "✓ Worst-tile visualizations created."
)


# ============================================================
# 17. VISUALIZE BEST TILES
# ============================================================

for rank, (_, row) in enumerate(
    best_tiles.head(10).iterrows(),
    start=1
):

    output_file = (
        VIS_DIR /
        f"best_{rank:02d}_{row['filename']}"
    )

    create_visualization(
        row,
        output_file,
        f"BEST #{rank}"
    )


print(
    "✓ Best-tile visualizations created."
)


# ============================================================
# 18. VISUALIZE COMPLETELY MISSED TILES
# ============================================================

for rank, (_, row) in enumerate(
    missed_tiles.head(20).iterrows(),
    start=1
):

    output_file = (
        VIS_DIR /
        f"missed_{rank:02d}_{row['filename']}"
    )

    create_visualization(
        row,
        output_file,
        f"COMPLETELY MISSED #{rank}"
    )


print(
    "✓ Missed-change visualizations created."
)


# ============================================================
# 19. VISUALIZE FALSE-CHANGE TILES
# ============================================================

for rank, (_, row) in enumerate(
    false_change_df.head(20).iterrows(),
    start=1
):

    output_file = (
        VIS_DIR /
        f"false_change_{rank:02d}_{row['filename']}"
    )

    create_visualization(
        row,
        output_file,
        f"FALSE CHANGE #{rank}"
    )


print(
    "✓ False-change visualizations created."
)


# ============================================================
# 20. SUMMARY STATISTICS
# ============================================================

summary = {

    "threshold":
        THRESHOLD,

    "total_tiles":
        len(tile_df),

    "change_tiles":
        int(
            (tile_df["GT_Changed_Pixels"] > 0)
            .sum()
        ),

    "no_change_tiles":
        int(
            (tile_df["GT_Changed_Pixels"] == 0)
            .sum()
        ),

    "completely_missed_tiles":
        int(
            tile_df["Completely_Missed"].sum()
        ),

    "false_change_tiles":
        int(
            tile_df["False_Change"].sum()
        ),

    "total_TP":
        total_tp,

    "total_TN":
        total_tn,

    "total_FP":
        total_fp,

    "total_FN":
        total_fn
}


summary_df = pd.DataFrame(
    [summary]
)


summary_df.to_csv(
    OUTPUT_DIR /
    "error_analysis_summary.csv",
    index=False
)


# ============================================================
# 21. FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 70)
print("ERROR ANALYSIS COMPLETE")
print("=" * 70)

print("\nResults saved to:")

print(
    OUTPUT_DIR
)

print("\nImportant files:")

print(
    "1. per_tile_analysis.csv"
)

print(
    "2. performance_by_change_size.csv"
)

print(
    "3. completely_missed_tiles.csv"
)

print(
    "4. false_change_tiles.csv"
)

print(
    "5. best_20_change_tiles.csv"
)

print(
    "6. worst_20_change_tiles.csv"
)

print(
    "7. overall_comparison.csv"
)

print(
    "8. VISUALIZATIONS/"
)

print("\n✓ Done.")

LEVIR-CD — DETAILED ERROR ANALYSIS

✓ All required files found.

Loading saved predictions...
Probabilities: (2048, 1, 256, 256)
Targets: (2048, 1, 256, 256)
Metadata: (2048, 5)

✓ Prediction / target / metadata counts match.


OVERALL TEST COMPARISON
 Threshold      IoU     Dice  Precision   Recall  Pixel_Accuracy  Predicted_Change_Percent  GT_Change_Percent
  0.500000 0.768747 0.869256   0.839666 0.901008        0.986193                  5.466425           5.094263
  0.850000 0.787377 0.881042   0.896164 0.866422        0.988081                  4.925191           5.094263


CALCULATING PER-TILE METRICS

✓ Per-tile analysis saved.


PERFORMANCE BY CHANGE SIZE
      Category  Tiles  Mean_IoU  Median_IoU  Mean_Dice  Mean_Precision  Mean_Recall  Mean_GT_Change_Percent  Mean_Pred_Change_Percent  Completely_Missed  False_Change
     No Change   1113    0.0000      0.0000     0.0000          0.0000       0.0000                  0.0000                    0.0334                  0           

In [2]:
# ============================================================
# LEVIR-CD — BLANK GROUND-TRUTH INVESTIGATION
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt


# ============================================================
# PATHS
# ============================================================

BASE_DIR = Path(
    r"C:\College\Dissertation\Data\FINAL_LEVIR_DATASET"
)

EVAL_DIR = (
    BASE_DIR
    / "CHANGE_DETECTION_MODEL"
    / "FINAL_TEST_EVALUATION"
)

CACHE_ROOT = (
    BASE_DIR
    / "CHANGE_DETECTION_MODEL"
    / "TEST_256_CACHE"
)

CACHE_A = CACHE_ROOT / "A"
CACHE_B = CACHE_ROOT / "B"
CACHE_MASK = CACHE_ROOT / "mask"

OUTPUT_DIR = (
    EVAL_DIR
    / "BLANK_GT_INVESTIGATION"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

VIS_DIR = (
    OUTPUT_DIR
    / "VISUALIZATIONS"
)

VIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# LOAD SAVED PREDICTIONS
# ============================================================

probabilities = np.load(
    EVAL_DIR /
    "test_probabilities.npy"
)

targets = np.load(
    EVAL_DIR /
    "test_targets.npy"
)

metadata = pd.read_csv(
    EVAL_DIR /
    "test_tile_metadata.csv"
)


print("=" * 70)
print("BLANK GROUND-TRUTH INVESTIGATION")
print("=" * 70)

print(
    "Total test tiles:",
    len(metadata)
)


# ============================================================
# PREPARE ARRAYS
# ============================================================

probs = probabilities[:, 0]

gt = targets[:, 0]


# Primary threshold selected from validation
THRESHOLD = 0.85


# ============================================================
# IDENTIFY BLANK GT TILES
# ============================================================

gt_changed_pixels = (
    gt > 0.5
).sum(
    axis=(1, 2)
)

blank_gt = (
    gt_changed_pixels == 0
)


blank_indices = np.where(
    blank_gt
)[0]


print(
    "\nBlank GT tiles:",
    len(blank_indices)
)

print(
    "Percentage of test tiles:",
    f"{len(blank_indices) / len(gt) * 100:.2f}%"
)


# ============================================================
# ANALYZE MODEL PREDICTIONS ON BLANK GT
# ============================================================

blank_rows = []


for idx in blank_indices:

    probability = probs[idx]

    prediction = (
        probability >= THRESHOLD
    )


    predicted_pixels = int(
        prediction.sum()
    )

    predicted_percent = (
        predicted_pixels /
        prediction.size *
        100
    )


    # Mean probability across the tile
    mean_probability = (
        probability.mean()
    )


    # Maximum probability
    max_probability = (
        probability.max()
    )


    # Number of pixels above several thresholds
    pixels_050 = int(
        (probability >= 0.50).sum()
    )

    pixels_070 = int(
        (probability >= 0.70).sum()
    )

    pixels_085 = int(
        (probability >= 0.85).sum()
    )

    pixels_095 = int(
        (probability >= 0.95).sum()
    )


    row = metadata.iloc[idx].to_dict()

    row.update({

        "prediction_threshold":
            THRESHOLD,

        "predicted_changed_pixels":
            predicted_pixels,

        "predicted_change_percent":
            predicted_percent,

        "mean_probability":
            mean_probability,

        "max_probability":
            max_probability,

        "pixels_prob_ge_050":
            pixels_050,

        "pixels_prob_ge_070":
            pixels_070,

        "pixels_prob_ge_085":
            pixels_085,

        "pixels_prob_ge_095":
            pixels_095
    })


    blank_rows.append(
        row
    )


blank_df = pd.DataFrame(
    blank_rows
)


# ============================================================
# SAVE ALL BLANK GT TILES
# ============================================================

blank_df = blank_df.sort_values(
    [
        "predicted_changed_pixels",
        "max_probability"
    ],
    ascending=False
)


blank_df.to_csv(
    OUTPUT_DIR /
    "all_blank_gt_tiles.csv",
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("MODEL BEHAVIOR ON BLANK GROUND TRUTHS")
print("=" * 70)


false_change_050 = (
    blank_df[
        blank_df["pixels_prob_ge_050"] > 0
    ]
)

false_change_085 = (
    blank_df[
        blank_df["predicted_changed_pixels"] > 0
    ]   
)


print(
    "\nBlank GT tiles:",
    len(blank_df)
)

print(
    "Blank GT + any prediction at 0.50:",
    len(false_change_050)
)

print(
    "Blank GT + prediction at 0.85:",
    len(false_change_085)
)


if len(blank_df) > 0:

    print(
        "\nPercentage with prediction at 0.85:",
        f"{len(false_change_085) / len(blank_df) * 100:.2f}%"
    )


# ============================================================
# STRONG PREDICTION THRESHOLDS
# ============================================================

for threshold in [
    0.50,
    0.70,
    0.85,
    0.95
]:

    count = (
        blank_df[
            blank_df[
                f"pixels_prob_ge_{int(threshold*100):03d}"
            ] > 0
        ]
        if f"pixels_prob_ge_{int(threshold*100):03d}"
        in blank_df.columns
        else 0
    )

    if isinstance(count, pd.DataFrame):

        print(
            f"Blank GT tiles with "
            f"≥1 pixel probability ≥ {threshold:.2f}:",
            len(count)
        )


# ============================================================
# TOP SUSPICIOUS TILES
# ============================================================

print("\n")
print("=" * 70)
print("TOP 30 BLANK-GT TILES WITH STRONGEST PREDICTIONS")
print("=" * 70)


top_suspicious = blank_df.head(
    30
)


print(
    top_suspicious[
        [
            "filename",
            "predicted_changed_pixels",
            "predicted_change_percent",
            "mean_probability",
            "max_probability"
        ]
    ].to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


top_suspicious.to_csv(
    OUTPUT_DIR /
    "top_30_suspicious_blank_gt_tiles.csv",
    index=False
)


# ============================================================
# CREATE VISUALIZATIONS
# ============================================================

print("\n")
print("=" * 70)
print("GENERATING VISUALIZATIONS")
print("=" * 70)


for rank, (_, row) in enumerate(
    top_suspicious.head(30).iterrows(),
    start=1
):

    filename = row["filename"]


    # --------------------------------------------------------
    # Find corresponding prediction index
    # --------------------------------------------------------

    matches = metadata.index[
        metadata["filename"]
        == filename
    ]

    if len(matches) == 0:
        continue

    idx = matches[0]


    # --------------------------------------------------------
    # Load images
    # --------------------------------------------------------

    img_a = np.asarray(
        Image.open(
            CACHE_A / filename
        ).convert("RGB")
    )


    img_b = np.asarray(
        Image.open(
            CACHE_B / filename
        ).convert("RGB")
    )


    gt_mask = np.asarray(
        Image.open(
            CACHE_MASK / filename
        ).convert("L")
    )


    gt_mask = (
        gt_mask > 127
    ).astype(np.uint8)


    probability = probs[idx]


    prediction = (
        probability >= THRESHOLD
    ).astype(np.uint8)


    # --------------------------------------------------------
    # Image difference
    # --------------------------------------------------------

    difference = np.abs(
        img_a.astype(np.float32)
        -
        img_b.astype(np.float32)
    )


    difference = (
        difference /
        max(
            difference.max(),
            1
        )
        * 255
    ).astype(np.uint8)


    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        1,
        5,
        figsize=(20, 4)
    )


    axes[0].imshow(
        img_a
    )

    axes[0].set_title(
        "Image A"
    )

    axes[0].axis(
        "off"
    )


    axes[1].imshow(
        img_b
    )

    axes[1].set_title(
        "Image B"
    )

    axes[1].axis(
        "off"
    )


    axes[2].imshow(
        gt_mask,
        cmap="gray"
    )

    axes[2].set_title(
        "Ground Truth\nBLANK"
    )

    axes[2].axis(
        "off"
    )


    axes[3].imshow(
        prediction,
        cmap="gray"
    )

    axes[3].set_title(
        f"Model Prediction\n"
        f"τ={THRESHOLD}"
    )

    axes[3].axis(
        "off"
    )


    axes[4].imshow(
        difference
    )

    axes[4].set_title(
        "Absolute A-B Difference"
    )

    axes[4].axis(
        "off"
    )


    fig.suptitle(
        f"Suspicious Blank GT #{rank} — "
        f"{filename}\n"
        f"Predicted change: "
        f"{row['predicted_change_percent']:.3f}% | "
        f"Max probability: "
        f"{row['max_probability']:.3f}",
        fontsize=12
    )


    plt.tight_layout()


    output_file = (
        VIS_DIR /
        f"{rank:02d}_{filename}"
    )


    plt.savefig(
        output_file,
        dpi=150,
        bbox_inches="tight"
    )


    plt.close()


print(
    "\n✓ Visualizations saved to:"
)

print(
    VIS_DIR
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("INVESTIGATION COMPLETE")
print("=" * 70)

print(
    "\nOpen this folder:"
)

print(
    VIS_DIR
)

print(
    "\nStart with the first 10 images."
)

print(
    "These are the blank-GT tiles where the model "
    "is most confident that change exists."
)

print(
    "\nIMPORTANT:"
)

print(
    "A blank GT does NOT automatically mean the label "
    "is wrong."
)

print(
    "We need to visually verify whether A and B "
    "actually contain a meaningful change."
)

BLANK GROUND-TRUTH INVESTIGATION
Total test tiles: 2048

Blank GT tiles: 1113
Percentage of test tiles: 54.35%


MODEL BEHAVIOR ON BLANK GROUND TRUTHS

Blank GT tiles: 1113
Blank GT + any prediction at 0.50: 56
Blank GT + prediction at 0.85: 43

Percentage with prediction at 0.85: 3.86%
Blank GT tiles with ≥1 pixel probability ≥ 0.50: 56
Blank GT tiles with ≥1 pixel probability ≥ 0.70: 50
Blank GT tiles with ≥1 pixel probability ≥ 0.85: 43
Blank GT tiles with ≥1 pixel probability ≥ 0.95: 37


TOP 30 BLANK-GT TILES WITH STRONGEST PREDICTIONS
            filename  predicted_changed_pixels  predicted_change_percent  mean_probability  max_probability
test_103_tile_15.png                     10947                 16.703796          0.180475         0.999608
 test_33_tile_01.png                      2190                  3.341675          0.046176         0.998384
test_116_tile_10.png                      1935                  2.952576          0.039642         0.999298
 test_86_tile_02.png 